# Day 3: Python for Web & Async (HTTPX, Coroutines, Generators & BeautifulSoup)

Welcome to Day 3 of **Learn Python in 5 Days**.

## What You Will Learn Today
- Managing web dependencies with **`uv add httpx beautifulsoup4`**
- Consuming REST APIs with **`HTTPX`** (both sync and async clients)
- Robust error handling: `raise_for_status()`, timeouts, and connection errors
- The Python **Async & Coroutine** mental model vs JavaScript Promises
- High-throughput parallel requests with **`asyncio.gather`**
- The iteration protocol: **Iterables vs. Iterators** (`iter()`, `next()`)
- Memory-efficient lazy streaming using **Generators (`yield`)**
- Web scraping and HTML data extraction with **`BeautifulSoup4`**
- Avoiding critical traps: un-awaited coroutines and blocking calls in event loops
- Practice Challenge: Building an Async API & Web Data Aggregator

---

## 1. Consuming REST APIs with Synchronous HTTPX
HTTPX is the modern standard HTTP client in Python (replacing legacy `requests` and `urllib`).

### Key Capabilities:
- `.get(url, params={...})`: Automatic query parameter encoding
- `.post(url, json={...})`: Automatic JSON serialization
- `.json()`: Direct parsing into Python dictionaries / lists
- `with httpx.Client() as client`: Connection pooling and HTTP keep-alive

In [ ]:
import httpx

# 1. Fetch public API data
response = httpx.get(
    "https://httpbin.org/get",
    params={"course": "learn-python", "day": 3},
    headers={"User-Agent": "PythonTrainingDemo/1.0"},
    timeout=10.0
)

print("Status Code:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))

# 2. Direct JSON parsing
data = response.json()
print("Echoed Query Args:", data.get("args"))

# 3. POST JSON payload
post_response = httpx.post(
    "https://httpbin.org/post",
    json={"task": "Learn Async Python", "status": "in-progress"}
)
print("Echoed JSON Payload:", post_response.json().get("json"))

## 2. HTTP Status Resilience: `raise_for_status()`
By default, HTTPX (like `fetch` in JavaScript) does not raise an exception when a server returns 404 or 500.
Calling **`response.raise_for_status()`** explicitly throws `httpx.HTTPStatusError` if the response indicates failure.

In [ ]:
def safe_fetch(url: str) -> dict | None:
    try:
        res = httpx.get(url, timeout=5.0)
        res.raise_for_status()  # Throws if 4xx or 5xx
        return res.json()
    except httpx.HTTPStatusError as err:
        print(f"[HTTP ERROR {err.response.status_code}] Failed to fetch {err.request.url}")
        return None
    except httpx.TimeoutException:
        print(f"[TIMEOUT] Request to {url} timed out")
        return None
    except httpx.RequestError as err:
        print(f"[CONNECTION ERROR] {err}")
        return None

# Test 404 handler
result = safe_fetch("https://httpbin.org/status/404")
print("Result of 404:", result)

## 3. Asynchronous Python: Coroutines vs JavaScript Promises
- **JavaScript:** Calling an `async` function immediately begins execution in the background.
- **Python:** Calling an `async def` function produces a **coroutine object** that remains idle until explicitly scheduled via `await` or `asyncio.gather()`.

> **Note on Jupyter:** In Jupyter Notebooks, an event loop is already running in the background. You can use top-level `await` directly in any cell!

In [ ]:
import asyncio

async def compute_latency(endpoint_name: str, delay_sec: float) -> str:
    print(f"[START] Pinging {endpoint_name}...")
    # In async code, use asyncio.sleep instead of time.sleep!
    await asyncio.sleep(delay_sec)
    print(f"[DONE] Received response from {endpoint_name}")
    return f"{endpoint_name}: {delay_sec * 1000:.0f}ms"

# Awaiting directly in notebook
result = await compute_latency("Auth API", 0.05)
print("Result:", result)

## 4. Async HTTP Client (`httpx.AsyncClient`)
Using `httpx.AsyncClient` within an `async with` context manager ensures TCP connection pooling and non-blocking I/O.

In [ ]:
async def fetch_ip_info() -> dict:
    async with httpx.AsyncClient(timeout=10.0) as client:
        res = await client.get("https://httpbin.org/ip")
        res.raise_for_status()
        return res.json()

ip_info = await fetch_ip_info()
print("Client IP Info:", ip_info)

## 5. Concurrent Network Requests with `asyncio.gather`
`asyncio.gather(*tasks)` is Python's direct equivalent of JavaScript's `Promise.all()`.
It fires all coroutines in parallel on the event loop and waits until all complete.

In [ ]:
import time

async def fetch_delay_endpoint(client: httpx.AsyncClient, delay: int) -> dict:
    res = await client.get(f"https://httpbin.org/delay/{delay}")
    return {"delay": delay, "url": str(res.url), "status": res.status_code}

async def run_batch():
    start_time = time.perf_counter()
    async with httpx.AsyncClient(timeout=10.0) as client:
        # Fire 3 requests in parallel: delays of 1s, 1s, and 1s
        tasks = [
            fetch_delay_endpoint(client, 1),
            fetch_delay_endpoint(client, 1),
            fetch_delay_endpoint(client, 1)
        ]
        results = await asyncio.gather(*tasks)
        
    total_time = time.perf_counter() - start_time
    print(f"Total execution time for 3 parallel requests: {total_time:.2f}s (Sync would take 3.0s!)")
    return results

batch_results = await run_batch()
print("Batch Results:", batch_results)

## 6. Iterables, Iterators, and the `iter()` / `next()` Protocol
- **Iterable:** Any object with `__iter__()` that can be passed to a `for` loop (e.g. lists, dicts, tuples, strings).
- **Iterator:** A stateful stream with `__next__()` that returns elements one by one until raising `StopIteration`.

In [ ]:
tech_stack = ["Python", "FastAPI", "Docker"]

# 1. Create an iterator
stack_iterator = iter(tech_stack)

# 2. Step forward manually with next()
print("Step 1:", next(stack_iterator))
print("Step 2:", next(stack_iterator))
print("Step 3:", next(stack_iterator))

# 3. Next call raises StopIteration (caught automatically by for loops)
try:
    next(stack_iterator)
except StopIteration:
    print("Iterator completely exhausted (StopIteration caught)")

## 7. Lazy Streaming with Generators (`yield`)
A generator function uses the **`yield`** keyword. Instead of allocating massive arrays in memory, it produces values lazily on demand.

In [ ]:
def stream_task_batches(total_tasks: int, batch_size: int = 2):
    """Generates batches of task IDs on demand without storing all in RAM."""
    for start_id in range(1, total_tasks + 1, batch_size):
        batch = list(range(start_id, min(start_id + batch_size, total_tasks + 1)))
        yield batch  # Pauses execution and yields control to the caller

# Consuming generator
for batch_num, batch in enumerate(stream_task_batches(7, batch_size=3), start=1):
    print(f"Batch #{batch_num}: Processing Task IDs {batch}")

## 8. Web Scraping with BeautifulSoup4
BeautifulSoup parses raw HTML and provides clean CSS selector queries (`select_one`, `select`, `find`, `find_all`).

In [ ]:
from bs4 import BeautifulSoup

sample_html = """
<!DOCTYPE html>
<html>
  <body>
    <div class="task-board">
      <div class="task-card" data-id="101">
        <h3 class="task-title">Migrate to FastAPI</h3>
        <span class="badge priority-high">High</span>
        <p class="desc">Upgrade legacy REST endpoints to FastAPI.</p>
        <a href="/tasks/101" class="details-link">View Details</a>
      </div>
      <div class="task-card" data-id="102">
        <h3 class="task-title">Dockerize Service</h3>
        <span class="badge priority-medium">Medium</span>
        <p class="desc">Create multi-stage Dockerfile.</p>
        <a href="/tasks/102" class="details-link">View Details</a>
      </div>
    </div>
  </body>
</html>
"""

soup = BeautifulSoup(sample_html, "html.parser")

# 1. Extract all task cards
cards = soup.select("div.task-card")
print(f"Found {len(cards)} task cards in DOM:\n")

extracted_tasks = []
for card in cards:
    task_id = int(card.get("data-id", 0))
    title = card.select_one("h3.task-title").get_text(strip=True)
    priority = card.select_one("span.badge").get_text(strip=True)
    link = card.select_one("a.details-link")["href"]
    
    extracted_tasks.append({"id": task_id, "title": title, "priority": priority, "link": link})

for t in extracted_tasks:
    print(f" - [#{t['id']}] {t['title']:<22} | Priority: {t['priority']:<6} | Link: {t['link']}")

---
## 9. Practice Challenge: Async API Aggregator
### Objective:
Build an asynchronous data pipeline combining HTTPX, `asyncio.gather`, and dataclasses:
1. Define a `@dataclass` named `ApiHealthMetric` with fields: `endpoint: str`, `status_code: int`, `latency_ms: float`, and `is_healthy: bool`.
2. Create an async function `ping_service(client: httpx.AsyncClient, url: str) -> ApiHealthMetric` that records latency and status.
3. Concurrently ping a list of 4 endpoints using `asyncio.gather()`.
4. Print a formatted health check report.

In [ ]:
# TODO: Write your challenge solution here...


---
## 10. Challenge Solution

In [ ]:
# Reference Solution
from dataclasses import dataclass
import asyncio
import httpx
import time

@dataclass
class ApiHealthMetric:
    endpoint: str
    status_code: int
    latency_ms: float
    is_healthy: bool

async def ping_service(client: httpx.AsyncClient, url: str) -> ApiHealthMetric:
    start = time.perf_counter()
    try:
        res = await client.get(url, timeout=5.0)
        latency = (time.perf_counter() - start) * 1000
        return ApiHealthMetric(
            endpoint=url,
            status_code=res.status_code,
            latency_ms=latency,
            is_healthy=res.status_code == 200
        )
    except (httpx.RequestError, httpx.TimeoutException):
        latency = (time.perf_counter() - start) * 1000
        return ApiHealthMetric(
            endpoint=url,
            status_code=0,
            latency_ms=latency,
            is_healthy=False
        )

async def run_health_checks():
    endpoints = [
        "https://httpbin.org/get",
        "https://httpbin.org/status/200",
        "https://httpbin.org/status/500",
        "https://httpbin.org/delay/1"
    ]
    
    async with httpx.AsyncClient() as client:
        tasks = [ping_service(client, url) for url in endpoints]
        metrics = await asyncio.gather(*tasks)
        return metrics

health_report = await run_health_checks()

print("=" * 75)
print(f"{'ENDPOINT':<36} | {'STATUS':<6} | {'LATENCY':<10} | {'HEALTH'}")
print("=" * 75)
for m in health_report:
    health_tag = "HEALTHY" if m.is_healthy else "DEGRADED"
    print(f"{m.endpoint:<36} | {m.status_code:<6} | {m.latency_ms:6.1f}ms   | {health_tag}")

---
## 11. Day 3 Recap & Bridge to Day 4

### What We Accomplished Today:
1. **Modern HTTPX:** Consumed REST APIs synchronously and asynchronously with automatic JSON serialization.
2. **Async Concurrency:** Understood Python coroutine mechanics and executed high-throughput batches using `asyncio.gather()`.
3. **Generators & Iteration:** Streamed data lazily with `yield` to eliminate unnecessary memory allocation.
4. **Web Extraction:** Extracted clean structured datasets from raw HTML using BeautifulSoup4.

### Looking Ahead to Day 4: FastAPI + Pydantic
Now that we know how to consume web APIs and write async code, tomorrow we start **building our own production web services**:
- The **FastAPI** application object, routers, and request lifecycle
- Automatic request validation and OpenAPI schemas with **Pydantic**
- Exposing async endpoints that query services cleanly
- Structuring a real-world Python API project